# Synthetic Data Generation for Rationale Equivalence

This notebook documents the full pipeline for generating synthetic training data used to train a **rationale equivalence classifier**.

The classifier's task is to determine whether two natural-language rationales convey the **same meaning** (equivalent) or a **different meaning** (unequivalent). This kind of labelled pair data is expensive to collect from humans at scale, so it is generated synthetically using a large language model.

---

## What problem does this solve?

The downstream model (a fine-tuned RoBERTa classifier) needs to learn to distinguish between:
- **Equivalent pairs**: two texts that express the same underlying claim or verdict.
- **Unequivalent pairs**: two texts where the meaning has changed

Rather than hand-labelling thousands of pairs, we use a capable LLM (Qwen 2.5) to produce the synthetic counterpart for each original rationale. The label (`equivalent` or `unequivalent`) is controlled deterministically by which prompt we give the model.

---

## Two iterations of the pipeline

| Version | Approach | Model | Key idea |
|---------|----------|-------|----------|
| **v1 – Original** | Zero-shot | Qwen 2.5-3B-Instruct-GPTQ-Int4 | Static system prompt, no examples |
| **v2 – Improved** | Few-shot | Qwen 2.5-3B-Instruct-GPTQ-Int4 | Dynamic prompts seeded with manually labelled human examples |

Both versions use the same model. The difference is entirely in the prompting strategy.

---
## 0. Imports and dependencies

In [ ]:
import argparse
import random

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# vLLM is a high-throughput inference engine for large language models.
# It handles batching, KV-cache management, and GPU memory allocation
# efficiently, which makes it practical to run generation over thousands
# of rows in a reasonable amount of time.
from vllm import LLM, SamplingParams

---
## 1. Source dataset

The input is a CSV file derived from the **HateCOT** dataset a collection of social media posts annotated for hate speech, each accompanied by a natural-language explanation (rationale) of why the post does or does not constitute hate speech.

Each row contains:
- `post` — the original social media text
- `explanation` — the rationale for the hate speech classification
- `label` — the hate speech verdict (`hate` / `not hate`)
- `domain` — which sub-dataset the example comes from (used later for domain-OOD evaluation)

The `explanation` column is the field we will be rewriting synthetically.

In [ ]:
# Path to the curated source file. Change this to point to your local copy.
INPUT_PATH = "hatecot_final_D3.csv"

# The column whose text we will rewrite. Every other column is preserved
# unchanged in the output, so the synthetic file remains aligned with the
# original metadata (post, label, domain, etc.).
TARGET_COLUMN = "explanation"

df = pd.read_csv(INPUT_PATH)

# Drop rows where the target column is empty — the model has nothing to
# rewrite and would produce degenerate outputs.
df = df[df[TARGET_COLUMN].notna()]
df = df[df[TARGET_COLUMN].astype(str).str.strip() != ""]

print(f"Total usable rows: {len(df)}")
print(df[["domain", "label"]].value_counts().sort_index())

### Sampling

Full-dataset generation is run on a GPU cluster. For development and inspection we sample a small subset.

In [ ]:
# Number of rows to process. Set to -1 to run the full dataset.
N_SAMPLES = 50
SEED = 42

if N_SAMPLES > 0:
    sample = df.sample(n=min(N_SAMPLES, len(df)), random_state=SEED).reset_index(drop=True)
else:
    sample = df.reset_index(drop=True)

print(f"Working with {len(sample)} rows.")
sample.head(3)

### Label assignment

Each row in the output will carry one of two labels:
- **`equivalent`** — the synthetic text should preserve the meaning of the original.
- **`unequivalent`** — the synthetic text should change the meaning in a detectable way.

Labels are assigned by simple alternation (even index → equivalent, odd index → unequivalent). This guarantees a **balanced 50 / 50 class split** regardless of dataset size, which is important for training a binary classifier without needing to re-weight classes.

In [ ]:
# Alternating assignment produces an exactly balanced dataset.
labels = [
    "equivalent" if i % 2 == 0 else "unequivalent"
    for i in range(len(sample))
]

print(f"equivalent:   {labels.count('equivalent')}")
print(f"unequivalent: {labels.count('unequivalent')}")

---
## 2. Model setup

The model is loaded **once** and reused across both generation pipelines. Both pipelines use identical inference settings — only the prompts differ.

**Model:** `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` (4-bit GPTQ, ~4 GB VRAM)

In [ ]:
# dtype=float16 is required on NVIDIA V100 GPUs, which do not support
# bfloat16 hardware instructions.
llm = LLM(
    model="Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4",
    tensor_parallel_size=1,
    max_model_len=2048,
    gpu_memory_utilization=0.85,
    dtype="float16",
    quantization="gptq",
    trust_remote_code=True,
)
tokenizer = llm.get_tokenizer()

# Shared sampling parameters used by both pipelines.
sampling_params = SamplingParams(temperature=0.7, top_p=0.9, max_tokens=512)

---
## 2. Pipeline v1 — Zero-Shot Reformulation

The first version of the pipeline instructs the model through **system prompts alone**, with no demonstrations. It rewrites hate speech rationales in two modes:

- **Equivalent** prompt: produce an exact paraphrase that preserves every claim, entity, and verdict.
- **Unequivalent** prompt: produce a rewrite where the core assertion is clearly different — typically by flipping a verdict, swapping an entity, or inverting a sentiment.

**Model:** `Qwen/Qwen2.5-3B-Instruct` (unquantized, ~6 GB VRAM)

In [ ]:
# --- EQUIVALENT system prompt ---
# The key constraints are: preserve meaning, tone, named entities, numbers,
# and intent. The instruction to output ONLY the reformulated sentence prevents
# the model from adding preamble like "Sure, here is the paraphrase:" which
# would contaminate the output.
SYSTEM_PROMPT_EQUIVALENT = (
    "You are a precise paraphrasing assistant. Rewrite the user's sentence in "
    "different words while preserving the exact meaning, tone, named entities, "
    "numbers, and intent. Do not add, omit, or infer information. Output ONLY "
    "the reformulated sentence, with no preamble, no quotes, no commentary."
)

# --- UNEQUIVALENT system prompt ---
# The model is given a menu of legal transformations rather than a free brief.
# This specificity is necessary: without it, the model tends to produce
# near-paraphrases that are technically different but not detectably so —
# the resulting pairs would be mislabelled. The concrete example at the end
# anchors the model's behaviour in the zero-shot setting.
SYSTEM_PROMPT_UNEQUIVALENT = (
    "You are a sentence-rewriting assistant. Your job is to produce a sentence "
    "whose MEANING is clearly DIFFERENT from the original. The output must NOT "
    "be a paraphrase, a summary, or a shortened version of the input. Keep the "
    "topic, style, and approximate length similar, but apply at least ONE of "
    "the following transformations so the assertion changes:\n"
    "  - flip a verdict, claim, or negation (e.g. 'qualifies' -> 'does not "
    "qualify', 'is hate speech' -> 'is not hate speech', 'yes' -> 'no')\n"
    "  - swap a key entity, target group, person, or category for a different "
    "one\n"
    "  - invert the sentiment or stance (positive <-> negative, supports <-> "
    "opposes)\n"
    "  - change a key fact, number, or attribute\n"
    "Example:\n"
    "  Original: The post does not direct abuse at any identity and is a "
    "neutral statement.\n"
    "  Rewritten: The post directs abuse at a specific identity and is a "
    "hateful statement.\n"
    "A reader comparing the two sentences must immediately see they assert "
    "different things. Output ONLY the rewritten sentence, with no preamble, "
    "no quotes, no commentary."
)

In [ ]:
def build_prompts(tokenizer, sentences: list[str], system_prompts: list[str]) -> list[str]:
    """
    Convert raw (sentence, system_prompt) pairs into the full prompt strings
    that vLLM will feed to the model.

    apply_chat_template inserts the model-specific special tokens
    (e.g. <|im_start|>, <|im_end|> for Qwen) and structures the
    system / user turns in the format the model was instruction-tuned on.
    Skipping this step would cause the model to ignore the system prompt
    or produce garbled output.
    """
    prompts = []
    for sentence, system in zip(sentences, system_prompts):
        messages = [
            {"role": "system", "content": system},
            {"role": "user",   "content": str(sentence)},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,  # appends the assistant turn opener
            )
        )
    return prompts

In [ ]:
system_prompts_v1 = [
    SYSTEM_PROMPT_EQUIVALENT if label == "equivalent" else SYSTEM_PROMPT_UNEQUIVALENT
    for label in labels
]

originals_v1 = sample[TARGET_COLUMN].astype(str).tolist()
prompts_v1   = build_prompts(tokenizer, originals_v1, system_prompts_v1)

outputs_v1   = llm.generate(prompts_v1, sampling_params, use_tqdm=True)
synthetic_v1 = [output.outputs[0].text.strip() for output in outputs_v1]

In [ ]:
# Build the output dataframe by copying all original columns, then appending
# the two new columns. Keeping every original column is important so that
# downstream code (e.g. the domain-OOD split in judge_finetune) can still
# access fields like `domain` and `post`.
output_v1 = sample.copy()
output_v1["synthetic"] = synthetic_v1
output_v1["meaning"]   = labels

output_v1.to_csv("reformulated_v1.csv", index=False)
print(f"Saved {len(output_v1)} rows to reformulated_v1.csv")

# Quick sanity check: inspect a few pairs side by side.
output_v1[[TARGET_COLUMN, "synthetic", "meaning"]].head(6)

---
## 3. Pipeline v2 — Few-Shot with Human Examples

The zero-shot pipeline produces grammatically correct rewrites, but they can feel mechanical and easy to distinguish from human-written text purely by style. For the classifier to generalise to real annotator comments, the training data should **resemble the distribution of human writing**.

Pipeline v2 addresses this by:
1. Shifting the task from *rewriting a rationale* to *writing a first-person human annotator comment*.
2. Seeding each system prompt with **3 randomly sampled examples** drawn from a small manually-labelled set of real human comments.

This few-shot approach nudges the model toward the informal, opinionated register that actual annotators use, rather than the neutral register of a paraphrase tool.

**Model:** `Qwen/Qwen2.5-3B-Instruct-GPTQ-Int4` — same model as v1.

> The 7B model weights occupy ~7.5 GB, leaving no headroom for the KV cache on a 9.3 GB GPU. The key contribution of v2 is the **prompting strategy**, not the model size, so reusing the 3B model is the right trade-off here.

In [ ]:
def load_fewshot_examples(manual_labels_path: str) -> dict:
    """
    Load the manually annotated CSV and split examples by label.

    The file has a header row followed by an additional descriptive row
    (skiprows=1 skips that second row so pandas reads only real data).
    Returns a dict with keys 'equivalent' and 'unequivalent', each holding
    a list of {post, human_comment} records used as few-shot demonstrations.
    """
    df = pd.read_csv(manual_labels_path, skiprows=1, header=0)

    # Normalise column names to the four fields we care about.
    df.columns = ["post", "explanation", "human_comment", "label"] + list(df.columns[4:])
    df = df[["post", "human_comment", "label"]].dropna(
        subset=["post", "human_comment", "label"]
    )

    # Normalise the label strings so capitalisation differences do not cause
    # examples to be silently dropped.
    df["label"] = df["label"].str.strip().str.lower()

    return {
        "equivalent":   df[df["label"] == "equivalent"][["post", "human_comment"]].to_dict("records"),
        "unequivalent": df[df["label"] == "unequivalent"][["post", "human_comment"]].to_dict("records"),
    }


fewshot = load_fewshot_examples("synthetic_data_generation_improved/manual_labelling.csv")

print(f"Few-shot pool — equivalent:   {len(fewshot['equivalent'])} examples")
print(f"Few-shot pool — unequivalent: {len(fewshot['unequivalent'])} examples")

In [ ]:
def _format_shots(examples: list[dict], n: int, seed: int) -> str:
    """
    Randomly sample n examples and format them as a numbered list
    that will be embedded directly in the system prompt.

    A fresh seed per row ensures that different rows see different
    subsets of the example pool, which reduces prompt repetition and
    makes the model less likely to copy-paste from the demonstrations.
    """
    rng = random.Random(seed)
    picked = rng.sample(examples, min(n, len(examples)))

    shots = ""
    for i, ex in enumerate(picked, 1):
        shots += (
            f"\nExample {i}:\n"
            f"  Post:    {ex['post'].strip()}\n"
            f"  Comment: {ex['human_comment'].strip()}\n"
        )
    return shots

In [ ]:
def build_system_prompt_equivalent(examples: list[dict], n_shots: int = 3, seed: int = 0) -> str:
    """
    System prompt for the EQUIVALENT case.

    The model is asked to write a first-person comment that supports the AI
    label — i.e. a comment whose message is consistent with the original
    rationale. The instruction to avoid referencing the AI prevents the model
    from simply restating the label rather than reasoning about the post.
    """
    shots = _format_shots(examples, n_shots, seed)
    return (
        "You are a human annotator labelling social media posts for hate speech. "
        "You will be given a post and its AI classification. "
        "Write a short, casual, first-person comment stating YOUR OWN view of the post "
        "that supports the AI classification. "
        "Write as if you just read the post yourself - do NOT reference the AI, "
        "do NOT say 'I agree', just state your own view directly about the post. "
        "1-3 sentences, informal tone.\n\n"
        "Here are examples of how human annotators write their comments:"
        f"{shots}\n"
        "---\n"
        "Now write your own comment. Do NOT copy from the examples above. "
        "Output ONLY your comment, nothing else."
    )


def build_system_prompt_unequivalent(examples: list[dict], n_shots: int = 3, seed: int = 0) -> str:
    """
    System prompt for the UNEQUIVALENT case.

    Identical in structure to the equivalent prompt, but asks the model to
    write a comment that DISAGREES with the AI label. This generates a
    human-register text whose meaning is opposed to the original rationale.
    """
    shots = _format_shots(examples, n_shots, seed)
    return (
        "You are a human annotator labelling social media posts for hate speech. "
        "You will be given a post and its AI classification. "
        "Write a short, casual, first-person comment stating YOUR OWN view of the post "
        "that DISAGREES with the AI classification. "
        "Write as if you just read the post yourself - do NOT reference the AI, "
        "do NOT say 'I disagree', just state your own view directly about the post. "
        "1-3 sentences, informal tone.\n\n"
        "Here are examples of how human annotators write their comments:"
        f"{shots}\n"
        "---\n"
        "Now write your own comment. Do NOT copy from the examples above. "
        "Output ONLY your comment, nothing else."
    )

In [ ]:
# For v2, the user message contains both the post text and the AI label,
# because the model needs to know what verdict to support or oppose.
# Concatenating them into a single user turn is simpler than using a
# multi-turn dialogue and produces equally good results in practice.
originals_v2 = (
    "Post: " + sample["post"].astype(str) +
    "\nAI classification: " + sample["label"].astype(str)
).tolist()

# Build one system prompt per row. The seed is set to the row index so
# each row gets a reproducible but distinct few-shot selection.
system_prompts_v2 = [
    build_system_prompt_equivalent(fewshot["equivalent"], seed=i)
    if label == "equivalent"
    else build_system_prompt_unequivalent(fewshot["unequivalent"], seed=i)
    for i, label in enumerate(labels)
]

# Inspect the system prompt for the first row to verify formatting.
print("--- System prompt for row 0 ---")
print(system_prompts_v2[0])

In [ ]:
prompts_v2   = build_prompts(tokenizer, originals_v2, system_prompts_v2)
outputs_v2   = llm.generate(prompts_v2, sampling_params, use_tqdm=True)
synthetic_v2 = [output.outputs[0].text.strip() for output in outputs_v2]

In [ ]:
output_v2 = sample.copy()
output_v2["synthetic"] = synthetic_v2
output_v2["meaning"]   = labels

output_v2.to_csv("reformulated_v2.csv", index=False)
print(f"Saved {len(output_v2)} rows to reformulated_v2.csv")

output_v2[["post", TARGET_COLUMN, "synthetic", "meaning"]].head(6)

---
## 4. Quality Validation

Generating data with a label does not guarantee the label is correct. Two complementary checks are used to validate the output:

| Check | What it measures | Expected outcome |
|-------|-----------------|------------------|
| **TF-IDF cosine similarity** | Surface-level lexical overlap between original and synthetic text | Equivalent pairs → high similarity; unequivalent pairs → lower similarity |
| **LLM flip-rate audit** | Whether unequivalent pairs actually change the verdict (semantic check) | ≥ 80% of unequivalent pairs should reach a detectably different conclusion |

Both checks complement each other: cosine similarity is fast but surface-level; the LLM audit is slower but semantically aware.

### 4.1 TF-IDF Cosine Similarity

A high cosine similarity between an equivalent pair means the rewrite shares most of the important vocabulary with the original — a good sign that meaning is preserved. A low cosine similarity for an unequivalent pair indicates that the key terms changed, consistent with a meaning flip.

In [ ]:
# Use v2 output for the quality checks; replace with output_v1 to audit
# the zero-shot pipeline instead.
df_check = output_v2.copy()

# TF-IDF vectors represent each text as a weighted bag of n-grams.
# Fitting the vectorizer on the union of both columns ensures both texts
# are projected into the same vocabulary space before comparing.
vectorizer = TfidfVectorizer()
all_texts  = df_check[TARGET_COLUMN].tolist() + df_check["synthetic"].tolist()
vectorizer.fit(all_texts)

originals_tfidf  = vectorizer.transform(df_check[TARGET_COLUMN])
synthetics_tfidf = vectorizer.transform(df_check["synthetic"])

# Compute the cosine similarity for each pair individually (diagonal of the
# full similarity matrix).
similarities = [
    cosine_similarity(originals_tfidf[i], synthetics_tfidf[i])[0][0]
    for i in range(len(df_check))
]
df_check["cosine_sim"] = similarities

# Report mean cosine similarity stratified by label.
print(df_check.groupby("meaning")["cosine_sim"].describe().round(3))

### 4.2 Semantic Flip-Rate Audit

Cosine similarity cannot detect meaning changes expressed through negation (`"is hate speech"` vs `"is not hate speech"` score almost identically). The flip-rate audit uses GPT-4o-mini as a semantic judge to answer: *do these two texts reach the same conclusion?*

A correctly generated unequivalent pair should cause the judge to respond `DIFFERENT`. If it responds `SAME`, the pair is mislabelled — the model failed to actually flip the meaning.

The target flip rate is **≥ 80%** — i.e. at least 8 out of 10 unequivalent pairs should be genuinely different in meaning.

In [ ]:
import openai

# The audit is run on a random subset of unequivalent pairs to keep API
# costs manageable. 30 samples provides sufficient signal for a rough
# quality estimate.
AUDIT_N = 30
AUDIT_SEED = 0

unequivalent_rows = df_check[df_check["meaning"] == "unequivalent"].sample(
    n=min(AUDIT_N, len(df_check[df_check["meaning"] == "unequivalent"])),
    random_state=AUDIT_SEED,
)


def judge(original: str, synthetic: str, client: openai.OpenAI) -> str:
    """
    Ask GPT-4o-mini whether two texts convey the same or different meaning.

    Returns 'SAME', 'DIFFERENT', or 'ERROR'. The prompt is intentionally
    minimal to reduce the chance the model hedges — we want a binary verdict.
    """
    prompt = (
        "Do the following two sentences convey the SAME or DIFFERENT meaning?\n"
        f"A: {original}\n"
        f"B: {synthetic}\n"
        "Reply with exactly one word: SAME or DIFFERENT."
    )
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
            temperature=0.0,  # deterministic output
        )
        return response.choices[0].message.content.strip().upper()
    except Exception as e:
        return "ERROR"


# Initialise the OpenAI client (requires OPENAI_API_KEY in the environment).
client = openai.OpenAI()

verdicts = [
    judge(row[TARGET_COLUMN], row["synthetic"], client)
    for _, row in unequivalent_rows.iterrows()
]

n_different = verdicts.count("DIFFERENT")
n_same      = verdicts.count("SAME")
n_error     = verdicts.count("ERROR")
flip_rate   = n_different / (n_different + n_same) if (n_different + n_same) > 0 else 0

print(f"Audited {len(verdicts)} unequivalent pairs")
print(f"  DIFFERENT (correct flip): {n_different}")
print(f"  SAME      (failed flip):  {n_same}")
print(f"  ERROR:                    {n_error}")
print(f"  Flip rate: {flip_rate:.1%}")

---
## 5. Summary and output schema

Both pipelines produce a CSV with the following columns:

| Column | Source | Description |
|--------|--------|-------------|
| `post` | original | The raw social media post |
| `explanation` | original | The human-written rationale |
| `label` | original | Hate speech verdict (`hate` / `not hate`) |
| `domain` | original | Sub-dataset identifier (used for domain-OOD splits) |
| `synthetic` | **generated** | The LLM-generated equivalent or unequivalent text |
| `meaning` | **generated** | `equivalent` or `unequivalent` |

This output is consumed directly by `judge_finetune/src/data.py`, which splits it by domain into train / val / test sets and tokenizes `(explanation, synthetic)` pairs for RoBERTa fine-tuning.

### Design decisions worth noting

- **Alternating label assignment** ensures a perfectly balanced 50/50 split without post-hoc resampling.
- **Per-row random seeds** for few-shot sampling prevent the model from seeing identical demonstrations across consecutive rows, which would reduce output diversity.
- **AWQ quantization** for the 7B model is a practical necessity on consumer-grade GPUs and introduces negligible quality degradation for this generation task.
- **The flip-rate audit** uses an LLM judge rather than rule-based heuristics because negation-based meaning changes (e.g. adding "not") are invisible to surface-level metrics like cosine similarity.